# BirdCLEF+ 2026 — Phase 1: Perch Embedding Extraction (Kaggle GPU)

用 Kaggle GPU 跑 Perch v2 GPU,對全部 `train_audio` 與 `train_soundscapes_labels` 的音訊抽 embedding,輸出供 Phase 2 linear probe 訓練使用。

## Kaggle Notebook 設定

- **Accelerator:** GPU P100(推薦)或 GPU T4 x2
- **Internet:** ON
- **Persistence:** Files only
- **Input:** Add Competitions → **BirdCLEF+ 2026**

預估時間:P100 約 30-60 分鐘(視第一次 XLA 暖機時間)。

---

## 執行流程

1. 跑 **Cell 1**(pip install)→ **Kernel → Restart**
2. 從 **Cell 2** 開始依序跑到 **Cell 8**
3. 最後 **Save Version**,Output 會成為下一個 notebook 的 Input

## Cell 1. 升級 TF + 裝 perch-hoplite

Kaggle 預裝 TF 2.19 的 StableHLO runtime 太舊,無法讀 Perch v2 的 `vhlo.cosine_v2` op,必須升到 2.20。

**⚠️ 跑完這個 cell 後必須 `Kernel → Restart & Clear Outputs`,再從 Cell 2 繼續。**

**若升級後 `import tensorflow` 出現 `undefined symbol` / `NotFoundError`**:Kaggle 的舊 .so 檔沒被清掉,Session → **Factory Reset** 再從頭來,並考慮改走 `perch_8` 後備路徑(Cell 3 說明)。

In [ ]:
!pip install -q -U "tensorflow==2.20.0" perch-hoplite

## Cell 2. 環境檢查 + 路徑

重啟 kernel 後從這裡開始。三個 assert 全部過才能往下。

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf
import numpy as np
import pandas as pd
import soundfile as sf
from pathlib import Path
from tqdm.auto import tqdm
import ast, gc, time

print("TF :", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

DATA_ROOT = Path("/kaggle/input/competitions/birdclef-2026")
WORK = Path("/kaggle/working")
WORK.mkdir(exist_ok=True)

assert tf.__version__.startswith("2.20"), f"需 TF 2.20,目前 {tf.__version__} — 請重啟 kernel"
assert tf.config.list_physical_devices("GPU"), "GPU 未偵測到 — 右側 Accelerator 選 P100/T4"
assert DATA_ROOT.exists(), "找不到比賽資料 — 右側 Add Data → Competitions 加入 BirdCLEF+ 2026"

## Cell 3. 載入 Perch 模型 + warm-up + 檢查 batchable

**Perch v2 有兩個變體,本 notebook 要用 GPU 版:**

| 模型 | 編譯平台 | 用在哪 |
|------|---------|--------|
| `perch_v2_gpu` | CUDA | **本 notebook(GPU 抽取)** |
| `perch_v2_cpu` | CPU | Phase 3 submission notebook(Kaggle 提交是 CPU) |

兩者**權重相同**,只是 XLA 目標平台不同。在 GPU 上跑 `perch_v2_cpu` 會報:
```
NotFoundError: The current platform CUDA is not among the platforms required by the module: [CPU]
```

**後備方案:** 若 TF 2.20 環境有問題,把 `MODEL_NAME` 改成 `"perch_8"`(Perch 1.0,1280 維,純 TF,TF 2.19 也能跑,LB ~0.85-0.88)。

In [ ]:
from perch_hoplite.zoo import model_configs

MODEL_NAME = "perch_v2_gpu"  # GPU 抽取用;submission 用 perch_v2_cpu;後備 perch_8

t0 = time.time()
model = model_configs.load_model_by_name(MODEL_NAME)
print(f"Model loaded in {time.time()-t0:.1f}s, sample_rate={model.sample_rate}")
print(f"model.batchable = {model.batchable}")

# Warm-up(單檔):觸發首次 XLA 編譯
t0 = time.time()
_ = model.embed(np.zeros(160_000, dtype="float32"))
print(f"Warm-up (single) in {time.time()-t0:.1f}s")

# Warm-up(batch=32):讓主迴圈要用的 size 預先編譯,避免進入 Cell 5 才重新編譯
t0 = time.time()
_ = model.batch_embed(np.zeros((32, 160_000), dtype="float32"))
print(f"Warm-up (batch=32) in {time.time()-t0:.1f}s")

assert model.batchable, "此模型不支援 batched 推論 — 請改用 perch_v2_gpu"

## Cell 4. 讀 metadata + label helpers

In [ ]:
taxonomy_df = pd.read_csv(DATA_ROOT / "taxonomy.csv")
train_df = pd.read_csv(DATA_ROOT / "train.csv")
ss_labels_df = pd.read_csv(DATA_ROOT / "train_soundscapes_labels.csv")

# train_soundscapes_labels.csv 的 start/end 欄位可能是 "HH:MM:SS" 字串,轉成浮點秒
if ss_labels_df["start"].dtype == object:
    ss_labels_df["start"] = pd.to_timedelta(ss_labels_df["start"]).dt.total_seconds()
    ss_labels_df["end"]   = pd.to_timedelta(ss_labels_df["end"]).dt.total_seconds()

species_list = taxonomy_df["primary_label"].tolist()
assert len(species_list) == 234
species_to_idx = {s: i for i, s in enumerate(species_list)}

print(f"train_audio rows    : {len(train_df):,}")
print(f"soundscape segments : {len(ss_labels_df):,}")
print(f"target classes      : {len(species_list)}")
print(f"class breakdown     : {taxonomy_df['class_name'].value_counts().to_dict()}")
print(f"start/end dtype     : {ss_labels_df['start'].dtype}  (應為 float64)")


def parse_secondary(s):
    """train.csv 的 secondary_labels 是字串化的 Python list。"""
    if not isinstance(s, str) or not s.strip():
        return []
    try:
        return ast.literal_eval(s)
    except Exception:
        return []


def multi_hot_from_row(row):
    """train_audio: primary + secondary → 234 維 multi-hot"""
    y = np.zeros(234, dtype="float32")
    if row["primary_label"] in species_to_idx:
        y[species_to_idx[row["primary_label"]]] = 1.0
    for s in parse_secondary(row.get("secondary_labels", "")):
        if s in species_to_idx:
            y[species_to_idx[s]] = 1.0
    return y


def multi_hot_from_soundscape(cell):
    """soundscape labels: 分號分隔多標籤 → 234 維 multi-hot"""
    y = np.zeros(234, dtype="float32")
    for s in str(cell).split(";"):
        s = s.strip()
        if s in species_to_idx:
            y[species_to_idx[s]] = 1.0
    return y

## Cell 5. 抽 train_audio embeddings(batched,主耗時)

**關鍵設計:** 所有檔案切成 5 秒 frame 後湊成固定 **BATCH_SIZE=32** 再送 GPU,避免 XLA 對每個不同 batch size 重新編譯。

**Perch 原生對齊:**
- 音檔 < 5 秒 → 補 0 到 5 秒(1 frame)
- 音檔 ≥ 5 秒 → 切成 `len // FRAME_LEN` 個完整 frame,末尾不足 5 秒的片段捨棄(與 `librosa.util.frame` 一致)

**進度觀察:** 第一個 batch 約 5-10 秒(warm-up 剩餘編譯)之後穩定應該是 **每 batch 0.3-1 秒**,整體 tqdm 速率 **20-50 files/sec**。若長期 < 2 files/sec,回去看 Cell 3 的 `model.batchable`。

In [ ]:
BATCH_SIZE = 32
FRAME_LEN = model.sample_rate * 5  # 160000

buf_audio, buf_labs, buf_meta = [], [], []
all_embs, all_labs, all_meta = [], [], []
fail_count = 0


def flush_buffer():
    if not buf_audio:
        return
    batch = np.stack(buf_audio, axis=0).astype("float32")  # (B, 160000)
    out = model.batch_embed(batch)
    embs = out.embeddings[:, 0, 0, :].astype("float32")     # (B, D)
    all_embs.append(embs)
    all_labs.append(np.stack(buf_labs).astype("float32"))
    all_meta.extend(buf_meta)
    buf_audio.clear()
    buf_labs.clear()
    buf_meta.clear()


for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="train_audio"):
    fp = DATA_ROOT / "train_audio" / row["filename"]
    try:
        audio, sr = sf.read(str(fp), dtype="float32")
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
    except Exception:
        fail_count += 1
        continue
    if len(audio) < FRAME_LEN:
        audio = np.pad(audio, (0, FRAME_LEN - len(audio)))
        n_frames = 1
    else:
        n_frames = len(audio) // FRAME_LEN
    y = multi_hot_from_row(row)
    for f_idx in range(n_frames):
        buf_audio.append(audio[f_idx * FRAME_LEN:(f_idx + 1) * FRAME_LEN])
        buf_labs.append(y)
        buf_meta.append((row["filename"], f_idx))
        if len(buf_audio) >= BATCH_SIZE:
            flush_buffer()
flush_buffer()  # 最後不足 BATCH_SIZE 的殘餘

X_train = np.concatenate(all_embs, axis=0)
Y_train = np.concatenate(all_labs, axis=0)
print(f"X_train={X_train.shape}, Y_train={Y_train.shape}, failed={fail_count}")

np.save(WORK / "train_audio_embeddings.npy", X_train)
np.save(WORK / "train_audio_labels.npy", Y_train)
pd.DataFrame(all_meta, columns=["filename", "frame_idx"]).to_parquet(WORK / "train_audio_meta.parquet")

del all_embs, all_labs
gc.collect()

## Cell 6. 抽 soundscape labeled segments

1,478 個 5 秒段。也用 batched 餵 GPU。讀過的 soundscape 檔快取,避免重複 IO。

In [ ]:
ss_cache = {}
ss_buf_audio, ss_buf_labs = [], []
ss_embs_list, ss_labs_list = [], []
fail = 0


def flush_ss_buffer():
    if not ss_buf_audio:
        return
    batch = np.stack(ss_buf_audio, axis=0).astype("float32")
    out = model.batch_embed(batch)
    ss_embs_list.append(out.embeddings[:, 0, 0, :].astype("float32"))
    ss_labs_list.append(np.stack(ss_buf_labs).astype("float32"))
    ss_buf_audio.clear()
    ss_buf_labs.clear()


for idx, row in tqdm(ss_labels_df.iterrows(), total=len(ss_labels_df), desc="soundscapes"):
    fn = row["filename"]
    if fn not in ss_cache:
        try:
            audio, sr = sf.read(str(DATA_ROOT / "train_soundscapes" / fn), dtype="float32")
            if audio.ndim > 1:
                audio = audio.mean(axis=1)
            ss_cache[fn] = audio
        except Exception:
            fail += 1
            continue
    audio = ss_cache[fn]
    s_samp = int(row["start"] * model.sample_rate)
    e_samp = int(row["end"] * model.sample_rate)
    seg = audio[s_samp:e_samp]
    if len(seg) < FRAME_LEN:
        seg = np.pad(seg, (0, FRAME_LEN - len(seg)))
    else:
        seg = seg[:FRAME_LEN]
    ss_buf_audio.append(seg)
    ss_buf_labs.append(multi_hot_from_soundscape(row["primary_label"]))
    if len(ss_buf_audio) >= BATCH_SIZE:
        flush_ss_buffer()
flush_ss_buffer()

X_ss = np.concatenate(ss_embs_list, axis=0)
Y_ss = np.concatenate(ss_labs_list, axis=0)
print(f"X_ss={X_ss.shape}, Y_ss={Y_ss.shape}, failed={fail}")

np.save(WORK / "soundscape_embeddings.npy", X_ss)
np.save(WORK / "soundscape_labels.npy", Y_ss)

del ss_cache
gc.collect()

## Cell 7. 驗證輸出

In [ ]:
print("== Output files ==")
for p in sorted(WORK.glob("*.npy")) + sorted(WORK.glob("*.parquet")):
    size_mb = p.stat().st_size / 1e6
    print(f"  {p.name:<38} {size_mb:>8.1f} MB")

assert X_train.shape[0] == Y_train.shape[0]
assert X_ss.shape[0] == Y_ss.shape[0]
assert X_train.shape[1] == X_ss.shape[1]

print("\n✓ Sanity OK")
print(f"  embedding dim                  = {X_train.shape[1]}")
print(f"  train_audio frames             = {X_train.shape[0]:,}")
print(f"  soundscape frames              = {X_ss.shape[0]:,}")
print(f"  avg labels/frame (train_audio) = {Y_train.sum(axis=1).mean():.2f}")
print(f"  avg labels/frame (soundscape)  = {Y_ss.sum(axis=1).mean():.2f}")

## Cell 8 — 完成,下一步

1. 右上角 **Save Version → Save & Run All (Commit)**
2. Commit 完成後,這個 notebook 的 Output 會成為可供其他 notebook 引用的 Dataset
3. Phase 2 (linear probe) 的 notebook 把這個 notebook 加為 Input 即可:
   ```python
   X = np.load("/kaggle/input/<本notebook名>/train_audio_embeddings.npy")
   ```

**輸出檔案:**
- `train_audio_embeddings.npy`   shape `(N, D)`  ~1-2 GB
- `train_audio_labels.npy`       shape `(N, 234)`  ~130 MB
- `train_audio_meta.parquet`     filename + frame_idx 對照
- `soundscape_embeddings.npy`    shape `(1478, D)`  ~9 MB
- `soundscape_labels.npy`        shape `(1478, 234)`  ~1.4 MB